###  Confidence Intervals for `ssim_motion`

We compute **three 95% confidence intervals** for the population mean of the `ssim_motion` metric:

1. Across **all methods** (AI + Classical)
2. Only **AI-based methods**
3. Only **Classical methods**

This analysis helps quantify the uncertainty around the average SSIM under different conditions and provides insight into how method type affects deblurring quality.

---

####  Formula used:

\[
\bar{x} \pm z_{\alpha/2} \cdot \frac{s}{\sqrt{n}}
\]

Where:
- \(\bar{x}\) = sample mean  
- \(s\) = sample standard deviation  
- \(n\) = number of observations  
- \(z_{\alpha/2}\) = 1.96 for 95% confidence level

---

####  Interpretation:

> We are 95% confident that the true population mean of `ssim_motion` lies within the calculated interval.

This means that if we repeated the experiment many times, 95% of the confidence intervals computed this way would contain the true mean.




In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Load your dataset
df = pd.read_parquet("idpa_dataset.parquet")

# Drop rows with missing SSIM or method type
df = df.dropna(subset=["ssim_motion", "method_type"])

# Function to compute confidence interval
def compute_ci(series, z=1.96):
    mean = series.mean()
    std = series.std(ddof=1)
    n = series.count()
    margin = z * (std / np.sqrt(n))
    return mean, (mean - margin, mean + margin)

# Overall
mean_all, ci_all = compute_ci(df["ssim_motion"])

# AI methods only
mean_ai, ci_ai = compute_ci(df[df["method_type"] == "ai"]["ssim_motion"])

# Classical methods only
mean_classic, ci_classic = compute_ci(df[df["method_type"] == "classical"]["ssim_motion"])

print("ALL:", mean_all, ci_all)
print("AI:", mean_ai, ci_ai)
print("CLASSIC:", mean_classic, ci_classic)



NameError: name 'data' is not defined

### Hypothesis Testing on `ssim_motion`

We perform three independent t-tests to check whether the average SSIM differs significantly between different groups.

#### Tests performed:
1. **AI vs Classical methods**  
   \(H_0\): The mean SSIM is the same for both groups  
   \(H_1\): The mean SSIM is different

2. **Motion blur angle > 180° vs ≤ 180°**  
   \(H_0\): No difference in SSIM means based on angle

3. **High vs Low Contrast (> 0.25 threshold)**  
   \(H_0\): No difference in SSIM means between high and low contrast groups

Each test uses a 5% significance level (α = 0.05). We reject \(H_0\) if the p-value < 0.05.



In [ ]:
from scipy.stats import ttest_ind

# Group 1: AI vs Classical
ssim_ai = df[df["method_type"] == "ai"]["ssim_motion"]
ssim_classic = df[df["method_type"] == "classical"]["ssim_motion"]
t1, p1 = ttest_ind(ssim_ai, ssim_classic, equal_var=False)

# Group 2: Motion angle > 180 vs ≤ 180
group1 = df[df["motion_angle"] > 180]["ssim_motion"]
group2 = df[df["motion_angle"] <= 180]["ssim_motion"]
t2, p2 = ttest_ind(group1, group2, equal_var=False)

# Group 3: Contrast > 0.25 vs ≤ 0.25
g1 = df[df["rms_contrast"] > 0.25]["ssim_motion"]
g2 = df[df["rms_contrast"] <= 0.25]["ssim_motion"]
t3, p3 = ttest_ind(g1, g2, equal_var=False)

print("AI vs CLASSICAL:", t1, p1)
print("Angle > 180 vs ≤ 180:", t2, p2)
print("Contrast > 0.25 vs ≤ 0.25:", t3, p3)



NameError: name 'df_ai' is not defined

### Multiple Linear Regression on `ssim_motion`

We build a model to predict the SSIM after motion blur deblurring using:

- `rms_contrast`
- `sobel_edge_strength`
- `motion_length`
- `motion_angle`

This helps us understand which image or blur properties influence SSIM.

The model is:

\[
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3 + \beta_4 x_4
\]

We also provide a prediction (with 95% interval) for a new image with specific characteristics.



In [ ]:
import statsmodels.formula.api as smf

# Remove missing values
df = df.dropna(subset=["rms_contrast", "sobel_edge_strength", "motion_length", "motion_angle"])

# Fit regression model
model = smf.ols("ssim_motion ~ rms_contrast + sobel_edge_strength + motion_length + motion_angle", data=df).fit()
print(model.summary())

# Predict for a new observation
new_obs = pd.DataFrame({
    "rms_contrast": [0.3],
    "sobel_edge_strength": [55],
    "motion_length": [15],
    "motion_angle": [90]
})

prediction = model.get_prediction(new_obs)
print(prediction.summary_frame(alpha=0.05))



NameError: name 'data' is not defined

### Correlation Heatmap

We visualize the pairwise correlation between the numeric features and `ssim_motion`. This helps to detect linear relationships that could inform model building.



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(df[["ssim_motion", "rms_contrast", "sobel_edge_strength", "motion_length", "motion_angle"]].corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()



NameError: name 'sns' is not defined

### Backward Feature Selection

We apply backward feature selection using linear regression to identify the most relevant predictors for `ssim_motion`. Features are removed one at a time until only those that improve the model remain.



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler

X = df[["rms_contrast", "sobel_edge_strength", "motion_length", "motion_angle"]]
y = df["ssim_motion"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model_lr = LinearRegression()
selector = SequentialFeatureSelector(model_lr, direction="backward")
selector.fit(X_scaled, y)

selected = X.columns[selector.get_support()]
print("Selected features:", selected.tolist())



### Scatter Plots with Linear Fit

We plot the relationship between `ssim_motion` and each predictor. A linear regression line is added to visually estimate the strength and direction of the relationship.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ["rms_contrast", "sobel_edge_strength", "motion_length", "motion_angle"]

for ax, feature in zip(axes.flatten(), features):
    sns.regplot(x=df[feature], y=df["ssim_motion"], ax=ax, line_kws={"color": "red"})
    ax.set_title(f"SSIM vs {feature}")

plt.tight_layout()
plt.show()

